<a href="https://colab.research.google.com/github/Radhakuchekar/Preparation/blob/pyspark/highest_grossing_products.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Reference : https://datalemur.com/questions/sql-highest-grossing
to identify the top two highest-grossing products within each category in the year 2022. The output should include the category, product, and total spend.

In [1]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q http://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
!tar xf spark-3.5.1-bin-hadoop3.tgz
!pip install -q findspark
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"
import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()
spark.conf.set("spark.sql.repl.eagerEval.enabled", True) # Property used to format output tables better
from pyspark.sql.functions import *
spark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, TimestampType
from pyspark.sql.functions import col

# Define the schema for the DataFrame
schema = StructType([
    StructField("category", StringType(), True),
    StructField("product", StringType(), True),
    StructField("user_id", IntegerType(), True),
    StructField("spend", FloatType(), True),
    StructField("transaction_date", StringType(), True)  # Assuming raw date is initially in string format
])

# Example input data
data = [
    ("appliance", "refrigerator", 165, 246.00, "12/26/2021 12:00:00"),
    ("appliance", "refrigerator", 123, 299.99, "03/02/2022 12:00:00"),
    ("appliance", "washing machine", 123, 219.80, "03/02/2022 12:00:00"),
    ("appliance", " machine", 123, 21.80, "03/02/2022 12:00:00"),
    ("electronics", "vacuum", 178, 152.00, "04/05/2022 12:00:00"),
    ("electronics", "wireless headset", 156, 249.90, "07/08/2022 12:00:00"),
    ("electronics", "vacuum", 145, 189.00, "07/15/2022 12:00:00")
]

# Create the DataFrame
df = spark.createDataFrame(data, schema)

# Show the DataFrame
df.show()


+-----------+----------------+-------+------+-------------------+
|   category|         product|user_id| spend|   transaction_date|
+-----------+----------------+-------+------+-------------------+
|  appliance|    refrigerator|    165| 246.0|12/26/2021 12:00:00|
|  appliance|    refrigerator|    123|299.99|03/02/2022 12:00:00|
|  appliance| washing machine|    123| 219.8|03/02/2022 12:00:00|
|  appliance|         machine|    123|  21.8|03/02/2022 12:00:00|
|electronics|          vacuum|    178| 152.0|04/05/2022 12:00:00|
|electronics|wireless headset|    156| 249.9|07/08/2022 12:00:00|
|electronics|          vacuum|    145| 189.0|07/15/2022 12:00:00|
+-----------+----------------+-------+------+-------------------+



In [3]:
df = df.withColumn("transaction_date", to_timestamp(col("transaction_date"), "MM/dd/yyyy HH:mm:ss"))

# Filter rows where the year in transaction_date is 2022
df_2022 = df.filter(date_format(col("transaction_date"), "yyyy") == "2022")

In [4]:
df_2022

category,product,user_id,spend,transaction_date
appliance,refrigerator,123,299.99,2022-03-02 12:00:00
appliance,washing machine,123,219.8,2022-03-02 12:00:00
appliance,machine,123,21.8,2022-03-02 12:00:00
electronics,vacuum,178,152.0,2022-04-05 12:00:00
electronics,wireless headset,156,249.9,2022-07-08 12:00:00
electronics,vacuum,145,189.0,2022-07-15 12:00:00


In [5]:
grouped_dF_2022= df_2022.groupBy(["category", "product"]).agg(sum(col("spend")).alias("total_spend"))

In [6]:
from  pyspark.sql import Window
window_spec = Window.partitionBy("category").orderBy(desc(col("total_spend")))

result = grouped_dF_2022.withColumn("rn", dense_rank().over(window_spec)).filter("rn <=2").select(["category", "product", "total_spend"])

In [7]:
result

category,product,total_spend
appliance,refrigerator,299.989990234375
appliance,washing machine,219.8000030517578
electronics,vacuum,341.0
electronics,wireless headset,249.89999389648438
